# Tripletex AI Agent — Manual Testing

Test the `/solve` endpoint locally or against the sandbox account.

## Setup
1. Run the server locally: `uv run uvicorn src.main:app --port 8000`
2. Or use the sandbox credentials from `.env` to test API calls directly.

In [ ]:
import httpx
import os
import json
from dotenv import load_dotenv

load_dotenv("../.env")

# Config
AGENT_URL = "http://localhost:8000"  # Local server
SANDBOX_BASE_URL = os.getenv("API_URL", "https://kkpqfuj-amager.tripletex.dev/v2")
SESSION_TOKEN = os.getenv("SESSION_TOKEN", "")

print(f"Sandbox URL: {SANDBOX_BASE_URL}")
print(f"Token: {SESSION_TOKEN[:20]}..." if SESSION_TOKEN else "No token found")

In [ ]:
def send_solve(prompt: str, files: list = None) -> dict:
    """Send a task to the local /solve endpoint."""
    payload = {
        "prompt": prompt,
        "files": files or [],
        "tripletex_credentials": {
            "base_url": SANDBOX_BASE_URL,
            "session_token": SESSION_TOKEN,
        },
    }
    with httpx.Client(timeout=300) as client:
        resp = client.post(f"{AGENT_URL}/solve", json=payload)
        print(f"Status: {resp.status_code}")
        print(f"Response: {resp.json()}")
        return resp.json()


def query_sandbox(path: str, params: dict = None) -> dict:
    """Query the sandbox Tripletex API directly to verify results."""
    with httpx.Client(timeout=30) as client:
        resp = client.get(
            f"{SANDBOX_BASE_URL}{path}",
            auth=("0", SESSION_TOKEN),
            params=params or {"fields": "*", "count": 100},
        )
        return resp.json()

## Test 1: Create Employee (Tier 1)

Simple task — create an employee with a name and email, and set them as administrator.

In [ ]:
# From the task docs example prompt
send_solve("Opprett en ansatt med navn Ola Nordmann, ola@example.org. Han skal være kontoadministrator.")

In [ ]:
# Verify: check employees in sandbox
employees = query_sandbox("/employee", {"fields": "id,firstName,lastName,email"})
print(json.dumps(employees, indent=2, ensure_ascii=False))

## Test 2: Create Customer (Tier 1)

Create a basic customer entity.

In [ ]:
send_solve("Opprett en kunde med navn Acme AS, epost post@acme.no, organisasjonsnummer 123456789.")

In [ ]:
# Verify
customers = query_sandbox("/customer", {"fields": "id,name,email,organizationNumber", "isCustomer": True})
print(json.dumps(customers, indent=2, ensure_ascii=False))

## Test 3: Create Invoice (Tier 2, multi-step)

This requires creating a customer, product, order with lines, and then the invoice.

In [ ]:
send_solve(
    "Opprett en faktura for kunden 'Bergen Consulting AS' (epost: faktura@bergenconsulting.no). "
    "Fakturaen skal inneholde 5 timer konsulenttjenester til 1500 kr per time ekskl. mva. "
    "Fakturadato er 2026-03-19 og forfallsdato er 2026-04-19."
)

In [ ]:
# Verify invoice was created
from datetime import date
today = date.today().isoformat()
invoices = query_sandbox("/invoice", {
    "invoiceDateFrom": "2026-01-01",
    "invoiceDateTo": "2026-12-31",
    "fields": "id,invoiceNumber,invoiceDate,customer(name),amount",
})
print(json.dumps(invoices, indent=2, ensure_ascii=False))

## Test 4: English Prompt

Verify the agent handles English prompts correctly.

In [ ]:
send_solve(
    "Create a new department called 'Engineering' with department number 100."
)

In [ ]:
# Verify
departments = query_sandbox("/department", {"fields": "id,name,departmentNumber"})
print(json.dumps(departments, indent=2, ensure_ascii=False))

## Test 5: Direct Sandbox API Exploration

Use these cells to explore the sandbox API directly (no agent involved).

In [ ]:
# List all employees
print("=== Employees ===")
print(json.dumps(query_sandbox("/employee", {"fields": "id,firstName,lastName,email"}), indent=2, ensure_ascii=False))

# List all customers
print("\n=== Customers ===")
print(json.dumps(query_sandbox("/customer", {"fields": "id,name,email", "isCustomer": True}), indent=2, ensure_ascii=False))

# List VAT types (useful for products/invoices)
print("\n=== VAT Types ===")
print(json.dumps(query_sandbox("/ledger/vatType", {"fields": "id,name,percentage"}), indent=2, ensure_ascii=False))

# List payment types (useful for registering payments)
print("\n=== Payment Types ===")
print(json.dumps(query_sandbox("/ledger/paymentType", {"fields": "id,description"}), indent=2, ensure_ascii=False))